# Top exit velo + top 5 fastest pitches

Two TrackMan CSVs in, two tables out.

In [1]:
import polars as pl
from pathlib import Path

pl.Config.set_tbl_cols(20)  # show every column in the tables

# the two TrackMan CSVs (swap in your own)
CSV_1 = Path("/Users/matt/Desktop/delispice/backend/notebooks/LiveBpPitching_2026-10-10T010108_verified.csv")
CSV_2 = Path("/Users/matt/Desktop/delispice/backend/notebooks/LiveBpPitching_2026-10-10T182801_verified.csv")

# force the numeric columns to floats so a mostly-empty column can't come in as text
NUM = ["RelSpeed", "InducedVertBreak", "HorzBreak", "SpinRate", "SpinAxis",
       "ExitSpeed", "Angle", "Distance"]

df = pl.concat(
    [pl.read_csv(p, infer_schema_length=None, schema_overrides={c: pl.Float64 for c in NUM})
     .rename({"HitType": "TaggedHitType"}, strict=False)  # Live BP files call it HitType
     for p in (CSV_1, CSV_2)],
    how="diagonal_relaxed",  # tolerate column order differences between files
)

# same hit types as the parquets: no contact (null here) -> "Undefined"
HIT_TYPES = ["GroundBall", "LineDrive", "FlyBall", "Popup", "Bunt"]
df = df.with_columns(
    pl.when(pl.col("TaggedHitType").is_in(HIT_TYPES))
    .then(pl.col("TaggedHitType"))
    .otherwise(pl.lit("Undefined"))
    .alias("TaggedHitType")
)

print(f"{df.height} pitches from {CSV_1.name} + {CSV_2.name}")
df["TaggedHitType"].value_counts(sort=True)

696 pitches from LiveBpPitching_2026-10-10T010108_verified.csv + LiveBpPitching_2026-10-10T182801_verified.csv


In [2]:
# hardest-hit ball across every hit type (no PitchCall in these files, so hit type marks the batted balls)
top_ev = (
    df.filter(pl.col("TaggedHitType") != "Undefined", pl.col("ExitSpeed").is_not_null())
    .sort("ExitSpeed", descending=True)
    .head(1)
    .select(
        "Batter",
        "Pitcher",
        pl.col("TaggedPitchType").alias("Pitch"),
        pl.col("RelSpeed").round(1).alias("Pitch Velo"),
        pl.col("ExitSpeed").round(1).alias("EV"),
        pl.col("Angle").round(1).alias("LA"),
        pl.col("Distance").round(0).alias("Distance"),
        pl.col("TaggedHitType").alias("Hit Type"),
    )
)
top_ev

ColumnNotFoundError: unable to find column "PitchCall"; valid columns: ["PitchNo", "Date", "Time", "Pitcher", "PitcherId", "PitcherThrows", "PitcherTeam", "PitcherSet", "TaggedPitchType", "PitchSession", "Flag", "RelSpeed", "VertRelAngle", "HorzRelAngle", "SpinRate", "SpinAxis", "Tilt", "RelHeight", "RelSide", "Extension", "VertBreak", "InducedVertBreak", "HorzBreak", "PlateLocHeight", "PlateLocSide", "ZoneSpeed", "VertApprAngle", "HorzApprAngle", "ZoneTime", "pfxx", "pfxz", "x0", "y0", "z0", "vx0", "vy0", "vz0", "ax0", "ay0", "az0", "PlayID", "CalibrationId", "EffVelocity", "PracticeType", "Device", "Direction", "BatterId", "Batter", "HitSpinRate", "HitType", "ExitSpeed", "BatterSide", "Angle", "PositionAt110X", "PositionAt110Y", "PositionAt110Z", "Distance", "LastTrackedDistance", "HangTime", "Bearing", "ContactPositionX", "ContactPositionY", "ContactPositionZ", "SpinAxis3dTransverseAngle", "SpinAxis3dLongitudinalAngle", "SpinAxis3dActiveSpinRate", "SpinAxis3dSpinEfficiency", "SpinAx

In [ ]:
# 5 fastest pitches with their movement (IVB / HB in inches) and spin
top_velo = (
    df.filter(pl.col("RelSpeed").is_not_null())
    .sort("RelSpeed", descending=True)
    .head(5)
    .select(
        "Pitcher",
        pl.col("TaggedPitchType").alias("Pitch"),
        pl.col("RelSpeed").round(1).alias("Velo"),
        pl.col("InducedVertBreak").round(1).alias("IVB"),
        pl.col("HorzBreak").round(1).alias("HB"),
        pl.col("SpinRate").round(0).alias("Spin"),
        pl.col("SpinAxis").round(0).alias("Axis"),
    )
)
top_velo